# Design Patterns for Machine Learning Platforms

Classic Gang-of-Four (GoF) patterns still apply when you build ML platforms: **creational** patterns tame exploding model constructors, **structural** patterns isolate monitoring and loading concerns, and **behavioral** patterns keep training and batch jobs composable. Below we use primitives from `agentic_assistants.core.patterns`.

## Introduction: GoF patterns in an ML context

Training pipelines, model registries, and online inference share problems with enterprise software: variation points (which model?), cross-cutting concerns (logging, caching), and orchestration (ordered steps with rollback). Mapping GoF patterns to ML makes those concerns explicit and testable.

In [ ]:
try:
    from agentic_assistants.core.patterns import (
        ModelFactory,
        PipelineBuilder,
        Subject,
        Observer,
        ProxyBase,
        LazyProxy,
        AlgorithmStrategy,
        StrategyContext,
        Command,
        CommandQueue,
    )
    PATTERNS_OK = True
except ImportError as e:
    PATTERNS_OK = False
    print("Import failed:", e)

PATTERNS_OK

## Creational: `ModelFactory` for dynamic model selection

Register constructors under string keys (e.g., from config or a router). The factory returns a fresh or configured instance without a giant `if/elif` chain.

In [ ]:
if PATTERNS_OK:
    class SyntheticRegressor:
        def __init__(self, name: str = "linear"):
            self.name = name
        def fit(self, X, y):
            return self
        def predict(self, X):
            return [0.0] * len(X)

    factory = ModelFactory()
    factory.register("linear", lambda: SyntheticRegressor("linear"))
    factory.register("dummy", lambda: SyntheticRegressor("dummy"))
    m = factory.create("linear")
    print("Created:", m.name, "registered:", factory.list_registered())

## Creational: `PipelineBuilder` for multi-stage pipelines

Add named steps that transform state; `build()` yields a `Pipeline` with `run()` / `async_run()`.

In [ ]:
if PATTERNS_OK:
    pipe = (
        PipelineBuilder()
        .add_step("load", lambda _: {"rows": 100})
        .add_step("transform", lambda s: {**s, "features": 12})
        .add_step("train", lambda s: {**s, "metric": 0.92})
        .configure("experiment", "demo")
    )
    result = pipe.run(None)
    print(result, "config", pipe.build().config)

## Structural: Observer / Subject for training monitoring

The `Subject` notifies `Observer` instances on events such as `epoch_end` or `checkpoint_saved`—useful for metrics loggers without coupling the trainer to every sink.

In [ ]:
if PATTERNS_OK:
    class MetricsLogger(Observer):
        def __init__(self):
            self.events = []
        def update(self, event: str, data=None):
            self.events.append((event, data))

    bus = Subject()
    log = MetricsLogger()
    bus.attach("epoch_end", log)
    bus.notify("epoch_end", {"epoch": 1, "loss": 0.4})
    bus.notify("epoch_end", {"epoch": 2, "loss": 0.31})
    print(log.events)

## Structural: `ProxyBase` and lazy loading

`ProxyBase` intercepts method calls on a wrapped subject (caching, auth). For **deferring** heavy model construction until first use, `LazyProxy` is the supplied helper; below we subclass `ProxyBase` to log the first `predict` while delegating to a lazily created model.

In [ ]:
if PATTERNS_OK:
    class HeavyModel:
        def predict(self, x):
            return x

    loads = {"n": 0}

    def make_model():
        loads["n"] += 1
        return HeavyModel()

    lazy = LazyProxy(make_model)
    _ = lazy.predict(1)
    _ = lazy.predict(2)
    print("LazyProxy factory calls:", loads["n"])

    class AuditingProxy(ProxyBase):
        def intercept(self, method_name, *args, **kwargs):
            print("call", method_name)
            fn = getattr(self._subject, method_name)
            return fn(*args, **kwargs)

    ap = AuditingProxy(HeavyModel())
    print(ap.predict(99))

## Behavioral: Strategy for algorithm swapping

`StrategyContext` delegates to an object implementing `fit` / `predict` (see `AlgorithmStrategy` protocol)—swap strategies at runtime (e.g., A/B or per-tenant config).

In [ ]:
if PATTERNS_OK:
    class StrategyA:
        def fit(self, X, y=None):
            pass
        def predict(self, X):
            return ["A"] * len(X)

    class StrategyB:
        def fit(self, X, y=None):
            pass
        def predict(self, X):
            return ["B"] * len(X)

    ctx: StrategyContext[AlgorithmStrategy] = StrategyContext(StrategyA())
    print(ctx.execute("predict", [1, 2, 3]))
    ctx.set_strategy(StrategyB())
    print(ctx.execute("predict", [1, 2]))

## Behavioral: Command and `CommandQueue` for pipeline orchestration

Encapsulate steps as `Command` objects, queue them, and `execute_all()` in order; optional `undo` supports rollback where safe.

In [ ]:
if PATTERNS_OK:
    class TrainStep(Command):
        def execute(self):
            return "trained"

    class EvalStep(Command):
        def execute(self):
            return "evaluated"

    q = CommandQueue()
    q.add(TrainStep())
    q.add(EvalStep())
    print(q.execute_all())

## Conclusion

- Use **ModelFactory** and **PipelineBuilder** at the boundaries of config and orchestration.
- Use **Subject/Observer** (or `EventBus` in the same module) to keep training loops thin.
- Use **LazyProxy** / **ProxyBase** subclasses for load and cross-cutting inference concerns.
- Use **StrategyContext** and **CommandQueue** when behavior or step lists must change without rewriting the driver.

Together, these patterns keep ML platforms easier to extend and reason about as the number of models and jobs grows.